In [1]:
import numpy as np
import pandas as pd

## Load & Merge data

In [2]:
pre_qual = pd.read_csv("../data/processed/pre_qualifying_dataset.csv")
race_raw = pd.read_csv("../data/raw/raw_race_data.csv")
quali_raw = pd.read_csv("../data/raw/raw_quali_data.csv")
fp2_raw = pd.read_csv("../data/raw/raw_fp2_data.csv")

In [3]:
df = pre_qual.copy()

race_merge = (
    race_raw[["season", "round", "driver", "team", "finish_position", "grid_position", "points", "status", "dnf_flag"]]
    .drop_duplicates(subset=["season", "round", "driver"])
)

df = df.merge(race_merge, on=["season", "round", "driver", "team"], how="left")

In [4]:
# Merge qualifying fields
# This weekend's qualifying results — grid position, gap to pole, wet flag

quali_merge = (
    quali_raw[[
        "season", "round", "driver",
        "quali_position",
        "gap_to_pole_seconds",
        "qualifying_session_was_dry",
    ]]
    .drop_duplicates(subset=["season", "round", "driver"])
)
df = df.merge(quali_merge, on=["season", "round", "driver"], how="left")

In [5]:
# Merge FP2 fields
fp2_merge = (
    fp2_raw[[
        "season", "round", "driver",
        "fp2_long_run_avg_gap",
        "fp2_session_was_dry",
    ]]
    .drop_duplicates(subset=["season", "round", "driver"])
)
df = df.merge(fp2_merge, on=["season", "round", "driver"], how="left")

In [6]:
# FP2 coverage report
fp2_coverage = len(fp2_raw) / max(len(race_raw), 1) * 100
print(f"  FP2 coverage    : {fp2_coverage:.1f}% of driver-rounds")

  FP2 coverage    : 26.7% of driver-rounds


### BLOCK 9: THIS WEEKEND QUALIFYING RESULT

In [7]:
# Compute actual grid size per race (handles 20 and 22 driver grids)
actual_grid_size = (
    df.groupby(["season", "round"])["grid_position"]
    .transform("count")
)
actual_midfield = (actual_grid_size / 2).round()

df["grid_position"] = pd.to_numeric(df["grid_position"], errors="coerce")

# if qualifying position also missing, use last place (20/22) as fallback
df["grid_position"] = df["grid_position"].fillna(actual_grid_size)

In [8]:
# gap_to_pole_seconds
df["gap_to_pole_seconds"] = pd.to_numeric(df["gap_to_pole_seconds"], errors="coerce")

# NaN source: qualifying data missing in FastF1, driver did not set a time
# Fix 1: fill with circuit median gap (similar conditions)
circuit_pole_med = df.groupby("circuit_id")["gap_to_pole_seconds"].transform("median")
df["gap_to_pole_seconds"] = df["gap_to_pole_seconds"].fillna(circuit_pole_med)

# Fix 2: global fallback — 1.0 second is typical midfield gap
df["gap_to_pole_seconds"] = df["gap_to_pole_seconds"].fillna(1.0)

In [9]:
# gap_to_teammate_quali_this_weekend
_team_q_avg = (
    df.groupby(["season", "round", "team"])["quali_position"].transform("mean")
)

df["gap_to_teammate_quali_this_weekend"] = df["quali_position"] - _team_q_avg
# NaN source: single-driver team, quali_position missing for both teammates
# Fix: 0 — neutral (no known advantage over teammate this weekend)
df["gap_to_teammate_quali_this_weekend"] = (
    df["gap_to_teammate_quali_this_weekend"].fillna(0.0)
)

In [10]:
# quali_position_vs_season_avg
# driver_avg_quali_position_this_season is from Block 3 — already NaN-free
df["quali_position_vs_season_avg"] = (
    df["quali_position"] - df["driver_avg_quali_position_this_season"]
)
# NaN source: quali_position missing for this race
# Fix: 0 — neutral (matched their season average)
df["quali_position_vs_season_avg"] = (
    df["quali_position_vs_season_avg"].fillna(0.0)
)

In [11]:
# qualifying_session_was_dry
# 1 - Dry (Assume)
df["qualifying_session_was_dry"] = (pd.to_numeric(df["qualifying_session_was_dry"], errors="coerce")
                                    .fillna(1) # assume dry if missing
                                    .astype(int))

In [12]:
# grid_penalty_applied
df["grid_penalty_applied"] = (
    (df["grid_position"] - df["quali_position"]) >= 3
).astype(int)
# NaN source: both grid and quali position were NaN before fill
# Both are now filled above — should be NaN-free
# Safety net:
df["grid_penalty_applied"] = df["grid_penalty_applied"].fillna(0).astype(int)

### BLOCK 10: FP2 LONG RUN

In [13]:
# fp2_session_was_dry
df["fp2_session_was_dry"] = (
    pd.to_numeric(df["fp2_session_was_dry"], errors="coerce")
    .fillna(1)        # assume dry if FP2 data unavailable
    .astype(int)
)

# fp2_long_run_avg_gap
df["fp2_long_run_avg_gap"] = pd.to_numeric(
    df["fp2_long_run_avg_gap"], errors="coerce"
)
# NaN source 1: FP2 session was wet — intentionally set to NaN
df["fp2_long_run_avg_gap"] = np.where(
    df["fp2_session_was_dry"] == 1,
    df["fp2_long_run_avg_gap"],
    np.nan,
)

# NaN source 2: driver did not complete enough long run laps (< 3)
# NaN source 3: FP2 session missing entirely from FastF1

In [14]:
# Fix 1: circuit median — same circuit, similar conditions
circuit_fp2_med = df.groupby("circuit_id")["fp2_long_run_avg_gap"].transform("median")
df["fp2_long_run_avg_gap"] = df["fp2_long_run_avg_gap"].fillna(circuit_fp2_med)
 
# Fix 2: season + circuit median — same circuit in adjacent seasons
season_circuit_med = df.groupby(["season", "circuit_id"])["fp2_long_run_avg_gap"].transform("median")
df["fp2_long_run_avg_gap"] = df["fp2_long_run_avg_gap"].fillna(season_circuit_med)
 
# Fix 3: global median — last resort
global_fp2_med = df["fp2_long_run_avg_gap"].median()
df["fp2_long_run_avg_gap"] = df["fp2_long_run_avg_gap"].fillna(
    global_fp2_med if pd.notna(global_fp2_med) else 0.5
)
 
remaining_fp2_nan = df["fp2_long_run_avg_gap"].isna().sum()
if remaining_fp2_nan > 0:
    print(f"{remaining_fp2_nan} rows still NaN after all fill levels — filling with 0.5")
    df["fp2_long_run_avg_gap"] = df["fp2_long_run_avg_gap"].fillna(0.5)

### BLOCK 11: RACE STRATEGY SIGNALS

In [15]:
compound_map = {
    "HARD": 0, "MEDIUM": 1, "SOFT": 2,
    "INTERMEDIATE": 3, "WET": 4
}
if "Compound" in race_raw.columns:
    cmp_raw = (
        race_raw[["season", "round", "driver", "Compound"]]
        .drop_duplicates(subset=["season", "round", "driver"])
    )
    df = df.merge(cmp_raw, on=["season", "round", "driver"], how="left")
    df["starting_tyre_compound_encoded"] = df["Compound"].map(compound_map)
    df.drop(columns=["Compound"], inplace=True)
else:
    df["starting_tyre_compound_encoded"] = np.nan

# NaN source: compound not recorded in FastF1, or Compound column absent
# Fix: circuit-round most common compound (majority of drivers)
compound_mode = (
    df.groupby(["season", "round"])["starting_tyre_compound_encoded"]
    .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else 1)
)
df["starting_tyre_compound_encoded"] = (
    df["starting_tyre_compound_encoded"].fillna(compound_mode)
)
# Final fallback: Medium = 1 (most common starting compound globally)
df["starting_tyre_compound_encoded"] = (
    df["starting_tyre_compound_encoded"].fillna(1).astype(int)
)

In [16]:
# tyre_compound_vs_majority
_majority = df.groupby(["season", "round"])["starting_tyre_compound_encoded"].transform(
    lambda x: x.mode().iloc[0] if not x.empty else 1
)
df["tyre_compound_vs_majority"] = (
    df["starting_tyre_compound_encoded"] != _majority
).astype(int)
# NaN source: both compound values now filled — should be NaN-free
df["tyre_compound_vs_majority"] = df["tyre_compound_vs_majority"].fillna(0).astype(int)

In [17]:
# pit_stop_avg_time_loss_at_circuit
# Extracted from static dict in extractor.py — attached to race rows
if "pit_stop_avg_time_loss_at_circuit" not in df.columns:
    if "pit_stop_avg_time_loss_at_circuit" in race_raw.columns:
        pit_lookup = (
            race_raw[["circuit_id", "pit_stop_avg_time_loss_at_circuit"]]
            .drop_duplicates("circuit_id")
        )
        df = df.merge(pit_lookup, on="circuit_id", how="left")
    else:
        df["pit_stop_avg_time_loss_at_circuit"] = np.nan
 
# NaN source: circuit not in PIT_STOP_DELTA dict in extractor.py
# Fix: circuit average → global average (21.5s)
circuit_pit_med = df.groupby("circuit_id")["pit_stop_avg_time_loss_at_circuit"].transform("median")
df["pit_stop_avg_time_loss_at_circuit"] = (
    df["pit_stop_avg_time_loss_at_circuit"].fillna(circuit_pit_med).fillna(21.5)
)

### BLOCK 12: GRID CONTEXT

In [18]:
def count_faster_teams_ahead(race_group):
    # For each driver, count drivers starting ahead with faster team pace.
    # Low  = clear path to front
    # High = driver trapped behind faster cars
    results = []
    for _, row in race_group.iterrows():
        grid = row["grid_position"]
        pace = row["team_avg_finish_last_3"]
        if pd.isna(grid) or pd.isna(pace):
            results.append(np.nan)
            continue
        ahead  = race_group[race_group["grid_position"] < grid]
        faster = ahead[ahead["team_avg_finish_last_3"] < pace]
        results.append(len(faster))
    return pd.Series(results, index=race_group.index)

In [19]:
df["drivers_ahead_with_higher_team_pace"] = (
    df.groupby(["season", "round"], group_keys=False)
    .apply(count_faster_teams_ahead)
)

# NaN source: grid_position or team_avg_finish_last_3 was NaN
# Both are now filled — but safety net below
# Fallback: grid_position - 1 (assume all cars ahead are faster)
_fallback = (df["grid_position"] - 1).clip(lower=0)
df["drivers_ahead_with_higher_team_pace"] = (
    df["drivers_ahead_with_higher_team_pace"].fillna(_fallback).fillna(0)
)

C:\Users\HP\AppData\Local\Temp\ipykernel_11656\4024550803.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(count_faster_teams_ahead)


###  DROP INTERMEDIATE COLUMNS

In [20]:
intermediate_cols = [
    "finish_position", "points", "status", "dnf_flag",
    "quali_position",
]
df.drop(columns=[c for c in intermediate_cols if c in df.columns], inplace=True)

### FINAL COLUMN ORDER AND SAVE

In [21]:
identity_cols = ["season", "round", "circuit_id", "driver", "team"]
 
pre_qual_features_cols = [
    "team_avg_finish_last_3",
    "team_avg_finish_last_5",
    "team_points_last_3",
    "team_points_this_season",
    "team_championship_position",
    "team_points_gap_to_leader",
    "team_dnf_rate_this_season",
    "team_finish_rate_this_season",
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_points_this_season",
    "driver_championship_position",
    "driver_points_gap_to_leader",
    "driver_dnf_rate_this_season",
    "driver_finish_rate_this_season",
    "driver_top5_rate_this_season",
    "driver_avg_quali_position_last_3",
    "driver_avg_quali_position_last_5",
    "driver_avg_quali_position_this_season",
    "driver_quali_to_finish_delta_last_5",
    "avg_quali_gap_to_teammate_this_season",
    "avg_finish_gap_to_teammate_this_season",
    "teammate_head_to_head_ratio_this_season",
    "driver_avg_finish_at_circuit_last5",
    "driver_best_finish_at_circuit_last5",
    "driver_podiums_at_circuit_last5",
    "driver_avg_quali_position_at_circuit_last5",
    "driver_dnf_rate_at_circuit",
    "driver_visits_to_circuit",
    "driver_circuit_vs_season_delta",
    "team_avg_finish_at_circuit_last5",
    "team_wins_at_circuit_last5",
    "team_podium_rate_at_circuit",
    "team_avg_quali_position_at_circuit_last5",
    "team_dnf_rate_at_circuit",
    "team_circuit_vs_season_delta",
    "circuit_type_encoded",
    "circuit_avg_overtakes_last3years",
    "circuit_safety_car_probability",
    "circuit_lap_count",
    "season_round_number",
    "races_remaining_this_season",
    "is_season_opener",
    "seasons_since_regulation_change",
]
 
post_qual_only_features_cols = [
    "grid_position",
    "gap_to_pole_seconds",
    "gap_to_teammate_quali_this_weekend",
    "quali_position_vs_season_avg",
    "qualifying_session_was_dry",
    "grid_penalty_applied",
    "fp2_long_run_avg_gap",
    "fp2_session_was_dry",
    "starting_tyre_compound_encoded",
    "tyre_compound_vs_majority",
    "pit_stop_avg_time_loss_at_circuit",
    "drivers_ahead_with_higher_team_pace",
]
 
POST_QUAL_FEATURES = pre_qual_features_cols + post_qual_only_features_cols
TARGET = "winner"
 
available = [c for c in POST_QUAL_FEATURES if c in df.columns]
missing   = [c for c in POST_QUAL_FEATURES if c not in df.columns]
if missing:
    print(f"\n  Missing features: {missing}")
 
final_cols = identity_cols + available + [TARGET]
df_out     = df[[c for c in final_cols if c in df.columns]].copy()
 
df_out.to_csv("../data/processed/post_qualifying_dataset.csv", index=False)

In [22]:
df.shape

(2168, 62)